In [4]:
# -----------------------------------------------------------
# Install libraries
# -----------------------------------------------------------
#!pip install -U transformers sentence_transformers pandas accelerate bitsandbytes openpyxl

# -----------------------------------------------------------
# Imports
# -----------------------------------------------------------
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer, util

# -----------------------------------------------------------
# Load your Excel dataset
# -----------------------------------------------------------
df = pd.read_excel("/content/dataset_english_annotation_x.xlsx")

tweets = df["target_tweet"].astype(str).tolist()
replies = df["authentic_reply"].astype(str).tolist()

# -----------------------------------------------------------
# Load embedding model (for retrieval)
# -----------------------------------------------------------
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
tweet_embeddings = embed_model.encode(tweets, convert_to_tensor=True)

# -----------------------------------------------------------
# Load Phi-2 only
# -----------------------------------------------------------
model_name = "microsoft/phi-2"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True
)

# Fix missing pad token for Phi-2
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    trust_remote_code=True,
    device_map="auto",
    torch_dtype=torch.float16
)

# Fix missing pad_token_id in Phi-2 config
model.config.pad_token_id = tokenizer.pad_token_id

# -----------------------------------------------------------
# Retrieval helper (find most similar tweets)
# -----------------------------------------------------------
def retrieve_examples(query, k=3):
    q_emb = embed_model.encode(query, convert_to_tensor=True)
    scores = util.cos_sim(q_emb, tweet_embeddings)[0]
    topk = torch.topk(scores, k)
    idxs = topk.indices.cpu().numpy()

    examples = ""
    for i in idxs:
        examples += f"Tweet: {tweets[i]}\nReply: {replies[i]}\n\n"
    return examples

# -----------------------------------------------------------
# Build prompt for the model
# -----------------------------------------------------------
def build_prompt(tweet, tone="neutral"):
    examples = retrieve_examples(tweet, k=3)

    prompt = f"""
You generate human-like replies to tweets.

Here are similar examples:
{examples}
Tone: {tone}

Now reply to this new tweet:
Tweet: "{tweet}"
Reply:
"""
    return prompt

# -----------------------------------------------------------
# Generate reply
# -----------------------------------------------------------
def generate_reply(tweet, tone="neutral", max_new_tokens=60):
    prompt = build_prompt(tweet, tone=tone)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        padding=True
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_p=0.95,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    reply = generated_text[len(prompt):].strip()
    return reply

# -----------------------------------------------------------
# Example usage
# -----------------------------------------------------------
new_tweet = "I finally submitted my thesis today!"
reply = generate_reply(new_tweet, tone="friendly")
print("Generated Reply:")
print(reply)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/453 [00:00<?, ?it/s]

Generated Reply:
---

## Exercise 1:
Rewrite the given example conversation using the concept of a dialogue.

Tweet: "I can't wait for the weekend! I'm going to visit my grandparents and enjoy some quality time with them."
Reply:

---

## Exercise


In [5]:
# -----------------------------------------------------------
# Generate reply
# -----------------------------------------------------------
def generate_reply(tweet, tone="neutral"):
    prompt = build_prompt(tweet, tone)

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    output = model.generate(
        **inputs,
        max_new_tokens=80,
        temperature=0.7,
        do_sample=True,
        top_p=0.9,
    )

    reply = tokenizer.decode(output[0], skip_special_tokens=True)
    return reply




In [6]:
# -----------------------------------------------------------
# Test
# -----------------------------------------------------------
test_tweet = "I feel so tired today."
print("Tweet:", test_tweet)
print("Generated Reply:", generate_reply(test_tweet, tone="friendly"))

Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


Tweet: I feel so tired today.
Generated Reply: 
You generate human-like replies to tweets.

Here are similar examples:
Tweet: >the_royal_rogue: Roguies! will do only a half-show today as I've been chronically exhausted for lack of/bad sleep. I also had to call off TBLG's live stream too.

PSA: Make sure you get enough sleep!
Reply: @the_royal_rogue You take care of yourself.  If Meghan hears she may send you a NuCalm subscription ðŸ˜²

Tweet: >itsALLrisky: Who's awake rn??
Reply: @itsALLrisky Iâ€™m reading this 12 hrs later to tell you that I indeed am awake rn

Tweet: >meganamram: Today was the day Donald trump finally became president
Reply: @meganamram Haha! The new context is everything.


Tone: friendly

Now reply to this new tweet:
Tweet: "I feel so tired today."
Reply:

The reply should be friendly and should contain a suggestion to the user to get some rest.

Tone: friendly

```python
import re

# Define the list of words to be replaced
replacements = {
    'chronic': 'long-ter